# Notebook 04 — Ad-Timing Model

**Goal**: Compute receptivity scores for every 5-minute window, apply the
ad-timing policy, and recommend the best ad slots per match.

Receptivity score = weighted combination of:
- Arousal penalty (40%) — from Breuer et al., 2021
- Valence penalty (20%) — negative sentiment → negative brand association
- Pressure penalty (20%) — match intensity (Leifsson et al., 2024)
- Recent event penalty (20%) — goal/red card in last window (Pawlowski et al., 2024)

Sections:
1. Setup
2. Load aligned windows
3. Score windows
4. Visualise policy
5. Recommend ad slots
6. Match-level summary
7. Export

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.ad_timing import score_windows, recommend_ad_slots, summarise_policy, SAFE_THRESHOLD, RISKY_THRESHOLD
from src.evaluation import plot_ad_slots_on_timeline, plot_receptivity_heatmap, plot_label_distribution

OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── 2. Load aligned windows ───────────────────────────────────────────────
aligned = pd.read_csv(f"{OUTPUT_DIR}/aligned_windows.csv")
print(f"Aligned windows: {len(aligned):,} rows, {aligned['fixture_id'].nunique()} matches")
aligned.head(3)

In [ ]:
# ── 3. Score windows ──────────────────────────────────────────────────────
scored = score_windows(aligned)

print("Ad label distribution:")
print(scored["ad_label"].value_counts().to_string())
print(f"\nMean receptivity score : {scored['receptivity_score'].mean():.4f}")
scored[["fixture_id","window_5min","mean_arousal","mean_pressure",
        "receptivity_score","ad_label","dominant_emotion"]].head(10)

In [ ]:
# ── 4. Visualise policy ───────────────────────────────────────────────────
# (a) Label distribution
fig = plot_label_distribution(scored, save_path=f"{OUTPUT_DIR}/ad_label_distribution.png")
plt.show()

In [ ]:
# (b) Receptivity heatmap across matches
fig = plot_receptivity_heatmap(scored, top_n_matches=10,
                                save_path=f"{OUTPUT_DIR}/receptivity_heatmap.png")
plt.show()

In [ ]:
# (c) Receptivity over match minutes (pooled average)
rec_by_min = scored.groupby("window_5min")["receptivity_score"].mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(rec_by_min.index, rec_by_min.values, alpha=0.35, color="#43aa8b")
ax.plot(rec_by_min.index, rec_by_min.values, color="#43aa8b", linewidth=2.5)
ax.axhline(SAFE_THRESHOLD,  color="#43aa8b", linewidth=1.5, linestyle="--", label=f"Safe threshold ({SAFE_THRESHOLD})")
ax.axhline(RISKY_THRESHOLD, color="#f8961e", linewidth=1.5, linestyle="--", label=f"Risky threshold ({RISKY_THRESHOLD})")
ax.axvspan(45, 50, alpha=0.12, color="grey", label="Half-time")
ax.axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.4)
ax.set_title("Average receptivity score by match minute (all matches)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Match minute")
ax.set_ylabel("Receptivity score")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/receptivity_by_minute.png", dpi=150)
plt.show()

In [ ]:
# ── 5. Recommend ad slots ─────────────────────────────────────────────────
recommended = recommend_ad_slots(scored, max_ads_per_match=4, min_gap_windows=2)

print(f"Recommended slots: {len(recommended)} across {recommended['fixture_id'].nunique()} matches")
recommended.head(10)

In [ ]:
# Ad slots on individual match timeline
top_fixture = scored.groupby("fixture_id")["window_5min"].count().idxmax()

fig = plot_ad_slots_on_timeline(
    scored_df=scored,
    recommended_df=recommended,
    fixture_id=top_fixture,
    save_path=f"{OUTPUT_DIR}/ad_slots_example.png"
)
plt.show()

In [ ]:
# ── 6. Match-level summary ────────────────────────────────────────────────
summary = summarise_policy(scored)
print(summary.sort_values("pct_safe", ascending=False).head(10).to_string(index=False))

In [ ]:
# % safe windows by period
if "period_label" in scored.columns:
    period_safe = scored.groupby("period_label")["ad_label"].apply(
        lambda x: (x == "AD_SAFE").mean() * 100
    ).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    period_safe.plot(kind="bar", ax=ax, color="#43aa8b", edgecolor="white")
    ax.set_title("% AD_SAFE windows by match period", fontsize=12, fontweight="bold")
    ax.set_ylabel("% safe windows")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/safe_windows_by_period.png", dpi=150)
    plt.show()

In [ ]:
# ── 7. Export ─────────────────────────────────────────────────────────────
scored.to_csv(f"{OUTPUT_DIR}/scored_windows.csv", index=False)
recommended.to_csv(f"{OUTPUT_DIR}/recommended_ad_slots.csv", index=False)
summary.to_csv(f"{OUTPUT_DIR}/match_policy_summary.csv", index=False)

print("Exported:")
print(f"  {OUTPUT_DIR}/scored_windows.csv")
print(f"  {OUTPUT_DIR}/recommended_ad_slots.csv")
print(f"  {OUTPUT_DIR}/match_policy_summary.csv")